<a href="https://colab.research.google.com/github/mikexcohen/Substack/blob/main/PCA_noise_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

|<h2>Substack post:</h2>|<h1><a href="htasdft" target="_blank">How sensitive is PCA to noise?</a></h1>|
|-|:-:|
|<h2><h2>|<h2>It depends.</h2>|
|<h2>Teacher:<h2>|<h1>Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h1>|

<br>

<i>Using the code without reading the post may lead to confusion or errors.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
### set figure colors for Substack post

# svg plots (higher-res)
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

plt.rcParams.update({
    'figure.facecolor': '#171717', #'#383838',# substack background color
    'figure.edgecolor': '#171717',#'#383838',#
    'axes.facecolor':   '#171717',#'#383838',#
    'axes.edgecolor':   '#DDE2F4',
    'axes.labelcolor':  '#DDE2F4',
    'xtick.color':      '#DDE2F4',
    'ytick.color':      '#DDE2F4',
    'text.color':       '#DDE2F4',
    'axes.spines.right': False,
    'axes.spines.top':   False,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold'
})

# **Figure 1: Simulating covariances**

In [ ]:
k = 40 # number of data features (and PCs)
nobs = k*10 # n observations

# vector of eigenvalue magnitudes
comps = np.exp(np.linspace(1.5,-2,k))
comps[:4] = np.exp(comps[:4]/2)

# create a random population covariance with structure
V = np.linalg.qr(np.random.randn(k,k))[0]
popC = V @ np.diag(comps) @ V.T

# draw data from the population covariance matrix
Xorig = np.random.multivariate_normal(np.zeros(k),popC,nobs)

# calculate its sample covariance
Corig = np.cov(Xorig,rowvar=False)

# plot eigenvalues
plt.figure(figsize=(10,4))
plt.plot(np.arange(k)+.15,np.sort(np.linalg.eigvalsh(Corig))[::-1],'gs-',markersize=6,markerfacecolor=[.7,.9,.7,.7],label='Sample')
plt.plot(np.arange(k)-.15,comps,'bo-',markerfacecolor=[.7,.7,.9,.7],markersize=6,label='Population')
plt.legend()
plt.gca().set(xlabel='Component',ylabel='Eigenvalue',title='Population and sample scree plots')

plt.tight_layout()
plt.savefig('eigvals_simulation.png')
plt.show()

# **Figure 2: Original and noisy data**

In [ ]:
# noise amount (fraction of std)
noisefactor = .5

# create a covariance matrix
V = np.linalg.qr(np.random.randn(k,k))[0]
popC = V @ np.diag(comps) @ V.T

# create a data matrix
Xorig = np.random.multivariate_normal(np.zeros(k),popC,nobs)

# add noise
noisestd = np.std(Xorig) * noisefactor
Xnoise = Xorig + np.random.normal(0,noisestd,Xorig.shape)

# covariance matrices
Corig = np.cov(Xorig,rowvar=False)
Cnoise = np.cov(Xnoise,rowvar=False)


fig, axs = plt.subplot_mosaic(
    mosaic =
    """01
    23""", width_ratios=[3,1],figsize=(8,4), layout='compressed' )

axs['0'].imshow(Xorig.T,aspect='auto')
axs['0'].set(xlabel='Observations',ylabel='Features',title='Original data')

axs['1'].imshow(Corig,cmap='plasma')
axs['1'].set(xlabel='Features',ylabel='Features',title='Original covariance')

axs['2'].imshow(Xnoise.T,aspect='auto')
axs['2'].set(xlabel='Observations',ylabel='Features',title='Noised data')

axs['3'].imshow(Cnoise,cmap='plasma')
axs['3'].set(xlabel='Features',ylabel='Features',title='Noised covariance')

plt.savefig('original_and_noisy_data.png')
plt.show()

# **Figure 3: Scatter plots of data and covariances**

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(8,4))

axs[0].plot(Xorig.flatten()[::10],Xnoise.flatten()[::10],'gs',markerfacecolor=[.7,.9,.7,.3])
axs[0].set(xlabel='Original',ylabel='Noised',
           title=f'Matrix elements\nr = {np.corrcoef(Xorig.flatten(),Xnoise.flatten())[0,1]:.3f}')
axs[0].axline((0,0),slope=1,color='r',linestyle='--',zorder=400)


axs[1].plot(np.diag(Corig),np.diag(Cnoise),'w^',markerfacecolor=[.9,.3,.9,.5],label='Diagonal (variances)')
axs[1].plot(Corig[np.triu_indices(k,1)],Cnoise[np.triu_indices(k,1)],'wo',markerfacecolor=[.3,.9,.9,.3],label='Off-diagonal (covars.)')
axs[1].legend()
axs[1].set(xlabel='Original',ylabel='Noised',
           title=f'Covariance elements\nr = {np.corrcoef(Corig[np.triu_indices(k)],Cnoise[np.triu_indices(k)])[0,1]:.3f}')
axs[1].axline((0,0),slope=1,color='r',linestyle='--',zorder=400)

plt.tight_layout()
plt.savefig('original_and_noisey_comparison.png')
plt.show()

# **Figure 4: Comparing eigensolutions**

In [ ]:
# original data: eigensolution and sort
evals,evecs = np.linalg.eigh(Corig)
sidx = np.argsort(evals)[::-1]
evalsOrig = evals[sidx]
evecsOrig = evecs[:,sidx]

# noisy data: eigendecomposition
evals,evecs = np.linalg.eigh(Cnoise)
sidx = np.argsort(evals)[::-1]
evalsNoise = evals[sidx]
evecsNoise = evecs[:,sidx]

# cosine similarity of eigenvectors
evec_rs = []
for i in range(k):
    evec_rs.append( abs(evecsOrig[:,i] @ evecsNoise[:,i]) )

# visualize
_,axs = plt.subplots(1,2,figsize=(8,3.5))

axs[0].plot(evalsOrig,evalsNoise,'wh',markerfacecolor=[.9,.7,.7],markersize=10)
axs[0].axline((evalsOrig.min(),evalsNoise.min()),slope=1,color='w',linestyle='--',linewidth=.5,zorder=-4)
axs[0].set(xlabel='Original',ylabel='Noised',
           title=f'Eigenvalues\nr = {np.corrcoef(evalsOrig,evalsNoise)[0,1]:.3f}')

axs[1].plot(evec_rs,'ws',markerfacecolor=[.7,.7,.9],markersize=10)
axs[1].set(xlabel='Sorted component',ylabel='Cosine similarity',ylim=[-.02,1.05],
          title='Eigenvectors')

plt.suptitle('Impact of noise on eigendecomposition stability')

plt.tight_layout()
plt.savefig('eigensolution_similarities.png')
plt.show()

# **Figure 6: Subspace affinity scores**

In [ ]:
plt.figure(figsize=(8,3))
for n in range(1,k+1):

  # projections
  projO = evecsOrig[:,:n]  @ evecsOrig[:,:n].T
  projN = evecsNoise[:,:n] @ evecsNoise[:,:n].T

  # affinity score
  a = np.linalg.norm(projO.T@projN)**2 / n

  # plot
  plt.plot(n,a,'wh',markerfacecolor=[.9,.7,.7],markersize=10)

# adjust the plot
plt.plot(range(1,k+1), np.arange(1,k+1)/k,'--',color='lightblue',zorder=-7)
plt.gca().set(xlabel='Subspace dimensions',xticks=range(1,k+1,2),ylabel='Normalized subspace affinity')

plt.tight_layout()
plt.savefig('eigensolution_affinities.png')
plt.show()

# **Figure 7: Stability in repeated examples**

In [ ]:
nExps = 250

eig_rs = np.zeros((nExps,k+2))
affinities = np.zeros((nExps,k))

for runi in range(nExps):

  # create a population covariance matrix
  V = np.linalg.qr(np.random.randn(k,k))[0]
  popC = V @ np.diag(comps) @ V.T

  # sample 1, its covariance, and eigensolution
  X1 = np.random.multivariate_normal(np.zeros(k),popC,nobs)
  C1 = np.cov(X1,rowvar=False)
  evals,evecs = np.linalg.eigh(C1)
  sidx = np.argsort(evals)[::-1]
  evals1 = evals[sidx]
  evecs1 = evecs[:,sidx]

  # repeat for sample 2
  X2 = np.random.multivariate_normal(np.zeros(k),popC,nobs)
  # X2 = X1 + np.random.normal(0,X1.std()*noisefactor,X1.shape) # uncomment for adding random noise to X1 instead of a new sample
  C2 = np.cov(X2,rowvar=False)
  evals,evecs = np.linalg.eigh(C2)
  sidx = np.argsort(evals)[::-1]
  evals2 = evals[sidx]
  evecs2 = evecs[:,sidx]


  # correlation of covariances
  eig_rs[runi,0] = np.corrcoef(C1[np.triu_indices(k)],C2[np.triu_indices(k)])[0,1]

  # correlation of eigenvalues
  eig_rs[runi,1] = np.corrcoef(evals1,evals2)[0,1]

  # cosine similiarity of eigenvectors
  for i in range(k):
      eig_rs[runi,i+2] = abs(evecs1[:,i] @ evecs2[:,i])

      # fyi, in case you want to try correlation:
      #eig_rs[runi,i+2] = abs(np.corrcoef(evecs1[:,i],evecs2[:,i])[0,1])

  # affinity scores
  for n in range(1,k+1):
    proj1 = evecs1[:,:n] @ evecs1[:,:n].T
    proj2 = evecs2[:,:n] @ evecs2[:,:n].T
    affinities[runi,n-1] = np.linalg.norm(proj1.T@proj2)**2 / n

In [ ]:
# visualize
fig,ax = plt.subplots(1,figsize=(12,3.5))

# plot the covariance correlations
ax.plot(np.random.normal(-2,.09,nExps),eig_rs[:,0],'.',color=[.9,.7,.7,.5],markersize=4)
ax.plot([-2.3,-1.7],np.full(2,eig_rs[:,0].mean()),color=[.9,.3,.07],linewidth=2,zorder=100)

# plot the eigenvalues correlations
ax.plot(np.random.normal(-1,.09,nExps),eig_rs[:,1],'.',color=[.7,.9,.7,.5],markersize=4)
ax.plot([-1.3,-.7],np.full(2,eig_rs[:,1].mean()),color=[.07,.9,.3],linewidth=2,zorder=100)

# plot the eigenvector similarities and subspace affinities
for i in range(k):
  ax.plot(np.random.normal(i,.09,nExps),eig_rs[:,i+2],'.',color=[.7,.7,.9,.5],markersize=4)
  ax.plot([i-.3,i+.3],np.full(2,np.median(eig_rs[:,i+2])),color=[.07,.9,.93],linewidth=2,zorder=100)


# violin plot for similarities
v = ax.violinplot(eig_rs,range(-2,k),widths=.7)
for vb in v['bodies']: vb.set_facecolor([.7,.7,.9])
v['cbars'].set_edgecolor([.7,.7,.7])
v['cmins'].set_edgecolor([.7,.7,.7])
v['cmaxes'].set_edgecolor([.7,.7,.7])


# adjust the axis labels etc
ax.set(xticks=np.concatenate(([-2,-1],range(0,k,2))),xticklabels=[r'$\mathbf{\Sigma}$',r'$\mathbf{\lambda}$'] + list(range(0,k,2)),
           xlim=[-2.5,k+.5],xlabel='Component',ylabel='Similarity (r or cosine)',title='Eigenvector stability in repeated experiments')

plt.tight_layout()
plt.savefig('demo2_similarities.png')
plt.show()

In [ ]:
# visualize
fig,ax = plt.subplots(1,figsize=(12,3))

# plot the eigenvector similarities and subspace affinities
for i in range(k):
  ax.plot(np.random.normal(i+1,.09,nExps),affinities[:,i],'.',color=[.9,.7,.7,.5],markersize=4)
  ax.plot([i+.7,i+1.3],np.full(2,np.median(affinities[:,i])),color=[1,.36,.25],linewidth=2,zorder=100)


# violin plot for affinities
v = ax.violinplot(affinities,range(1,k+1),widths=.7)
for vb in v['bodies']: vb.set_facecolor([.9,.7,.7])
v['cbars'].set_edgecolor([.7,.7,.7])
v['cmins'].set_edgecolor([.7,.7,.7])
v['cmaxes'].set_edgecolor([.7,.7,.7])

ax.plot(range(1,k+1), np.arange(1,k+1)/k,'--',color='lightblue',label='Chance expectation')
ax.legend()

# adjust the axis labels etc
ax.set(xticks=range(1,k+1,2),xlabel='Subspace dimensions',ylabel='Normalized squared affinity',
           xlim=[-.5,k+.5],title='Subspace overlap in repeated experiments')

plt.tight_layout()
plt.savefig('demo2_affinities.png')
plt.show()